In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# KEEP COLAB ALIVE — clicks the "Connect" button every 60s in the
# background so Colab's ~90-min idle disconnect never triggers just
# from you not touching the browser tab while the bot runs.
#
# Note this only stops the IDLE timeout. Colab's free tier also has a
# hard ceiling on total session length (historically ~12h) that this
# trick can't get around — if a 12h run gets cut off right around that
# mark, that's the hard cap, not idle disconnect, and Colab Pro/Pro+
# raises it. Run this cell once, right after connecting.
from IPython.display import Javascript

display(Javascript('''
function KeepColabAlive(){
    console.log("[keepalive] ping");
    var btn = document.querySelector("colab-toolbar-button#connect")
           || document.querySelector("colab-connect-button")
           || document.querySelector("#top-toolbar colab-connect-button");
    if (btn) { btn.click(); }
}
if (window._keepAliveInterval) { clearInterval(window._keepAliveInterval); }
window._keepAliveInterval = setInterval(KeepColabAlive, 60000);
console.log("[keepalive] started");
'''))
print("[keepalive] cell running — pings the Connect button every 60s")


In [ ]:
#@title Bot Source (click to expand)
# ============================================================
# Commod15min V35a — PAPER ONLY (no order is ever sent to Kalshi)
#
# STRATEGY
#   - BUY window: minute 0 to minute 9 of each 15-minute market (900s to 360s
#     before close; from market open). All 11 markets.
#   - BUY: when a side's ask is 48-52c, buy 10 contracts of it at the ask. If both
#     sides are in range, buy the cheaper one (equal prices: wait).
#   - SELL: when that side's bid reaches 72c or more, sell all 10 at the bid.
#     Selling continues after the buy window closes (any time before close).
#   - If it never reaches 72c, it's held to settlement ($1 or $0).
#   - No buy when the bid-ask spread is wider than 5c (MAX_SPREAD_C).
#   - Repeat: after a sale, the same market can be bought again if a side
#     is in 47-53c again while the buy window is still open.
#     One open position per market at a time. No stops, no reversals.
#
# FILLS (paper, kept as close to live as possible):
#   - Buys fill at the current ASK, sells at the current BID (what a real
#     order pays/gets). Kalshi taker fee on every buy and sell:
#     0.07 x contracts x P x (1-P), rounded up to the cent, per order.
#   - The price feed has no order sizes, so a 10-contract order is assumed to
#     fill at the best price. Live fills could be a cent worse when the book
#     is thin.
#
# LOGS (Google Drive):
#   commod15min_v35a_trades.csv       one row per position (sold, or held to settlement)
#   commod15min_v35a_ticks.csv        every price update, all 11 markets, full 15 min
#   commod15min_v35a_prints.csv       every real trade print on Kalshi
#   commod15min_v35a_settlements.csv  Kalshi's official result for every market
# ============================================================
import os, sys, csv, json, time, math, base64, threading, datetime as dt
from concurrent.futures import ThreadPoolExecutor
import requests
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding
try:
    import websocket  # websocket-client package
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "websocket-client"], check=True)
    import websocket


# ---------------- settings ----------------
BOT_NAME  = "Commod15min V35a"
PEM_PATH  = '/content/drive/MyDrive/intraday key.pem'
KEY_ID    = "e0d95d64-fa0e-4c9e-be91-35660b0af725"
DRIVE     = '/content/drive/MyDrive/'
TRADES_CSV = DRIVE + 'commod15min_v35a_trades.csv'
TICK_CSV   = DRIVE + 'commod15min_v35a_ticks.csv'
PRINTS_CSV = DRIVE + 'commod15min_v35a_prints.csv'
SETTLE_CSV = DRIVE + 'commod15min_v35a_settlements.csv'

SERIES = ["KXBTC15M", "KXETH15M", "KXSOL15M", "KXXRP15M", "KXDOGE15M",
          "KXGOLD15M", "KXSILVER15M", "KXWTI15M", "KXCOPPER15M",
          "KXPLATINUM15M", "KXPALLADIUM15M"]

BUY_FROM_SEC  = 900    # buy window: from market open (900s before close) ...
BUY_UNTIL_SEC = 360    # ... until 360s before close (= the first 9 minutes)
BUY_MIN_C     = 48     # buy a side when its ask is 48c ...
BUY_MAX_C     = 52     # ... to 52c
SELL_AT_C     = 72     # sell when the held side's bid is 72c or more
CONTRACTS     = 10
MAX_SPREAD_C  = 5      # skip a buy when ask - bid is wider than this (empty book at market open)
DEADBAND_SEC  = 2      # never act this close to the close
MARKETS_REFRESH_SEC = 5
LOOP_SEC      = 0.5
SETTLE_RETRY_SEC  = 10
SETTLE_GIVEUP_SEC = 1800

BASE    = "https://api.elections.kalshi.com/trade-api/v2"
PREF    = "/trade-api/v2"
WS_URL  = "wss://external-api-ws.kalshi.com/trade-api/ws/v2"
WS_PATH = "/trade-api/ws/v2"

TRADE_COLS = ["window", "asset", "mode", "trade_no", "side", "buy_iso", "buy_s2c", "buy_px",
              "sell_iso", "sell_s2c", "sell_px", "result", "reason", "lowest_bid_held",
              "fees_c", "net_c", "run_net_c"]
TICK_COLS   = ["ts_iso", "ticker", "asset", "yes_bid", "yes_ask", "sec_to_close"]
PRINT_COLS  = ["ts_iso", "ticker", "asset", "yes_price", "count", "taker_side", "sec_to_close"]
SETTLE_COLS = ["logged_iso", "ticker", "asset", "close_time", "result", "floor_strike",
               "expiration_value", "status"]

# ---------------- helpers ----------------
def asset(t): return t.split("15M")[0].replace("KX", "")
def series_of(t): return t.split("-")[0]
def now(): return dt.datetime.now(dt.timezone.utc)
def now_iso(): return now().isoformat()
def parse_iso(s):
    try: return dt.datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    except (ValueError, TypeError): return None

def taker_fee_c(n, price_c):
    """Kalshi taker fee for one order: 0.07 x contracts x P x (1-P), rounded up to the cent."""
    return math.ceil(0.07 * n * price_c * (100 - price_c) / 100 - 1e-9)

def px_c(m, name):
    """Price field in cents. Kalshi sends '<name>_dollars' ("0.8900") and/or
    '<name>' in whole cents (89)."""
    v = m.get(name + "_dollars")
    if v is not None:
        try: return round(float(v) * 100)
        except (TypeError, ValueError): pass
    v = m.get(name)
    if v is None: return None
    try: return int(round(float(v)))
    except (TypeError, ValueError): return None

def load_key():
    p = os.path.expanduser(PEM_PATH)
    if not os.path.exists(p): sys.exit(f"PEM not found: {p}")
    return serialization.load_pem_private_key(open(p, "rb").read(), password=None)

def _sign(k, ts, method, path):
    return base64.b64encode(k.sign(f"{ts}{method}{path}".encode(),
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
        hashes.SHA256())).decode()

def _hdrs(k, method, path):
    ts = str(int(time.time() * 1000))
    return {"KALSHI-ACCESS-KEY": KEY_ID, "KALSHI-ACCESS-SIGNATURE": _sign(k, ts, method, PREF + path),
            "KALSHI-ACCESS-TIMESTAMP": ts, "Accept": "application/json"}

def api_get(sess, k, path, params=None):
    for attempt in range(3):
        try:
            r = sess.get(BASE + path, headers=_hdrs(k, "GET", path), params=params, timeout=8)
            if r.status_code == 404: return None
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(0.5 * (attempt + 1)); continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            time.sleep(0.5 * (attempt + 1))
    return None

def _fetch_series_open(sess, k, ser):
    d = api_get(sess, k, "/markets", {"series_ticker": ser, "status": "open", "limit": 100}) or {}
    return d.get("markets", []) or []

def fetch_open(sess, k):
    out = []
    with ThreadPoolExecutor(max_workers=len(SERIES)) as ex:
        for res in ex.map(lambda s: _fetch_series_open(sess, k, s), SERIES):
            out += res
    return out

def fetch_market(sess, k, ticker):
    d = api_get(sess, k, f"/markets/{ticker}")
    return (d or {}).get("market", {}) if d else {}

# ---------------- Kalshi websocket: prices + trade prints ----------------
class PriceFeed:
    CHANNELS = ("ticker", "trade")

    def __init__(self, k):
        self.k = k
        self.prices = {}
        self.lock = threading.RLock()
        self.send_lock = threading.Lock()
        self.tracked = set()
        self.subscribed = set()
        self.sub_pending = False
        self.sids = {}
        self.ws = None
        self.ready = threading.Event()
        self.stop_flag = threading.Event()
        self._next_id = 1
        self.connect_count = 0
        self.on_tick = None    # on_tick(ticker, yes_bid, yes_ask)
        self.on_trade = None   # on_trade(ticker, yes_price, count, taker_side)

    def _cmd_id(self):
        with self.lock:
            i = self._next_id; self._next_id += 1
        return i

    def _headers(self):
        ts = str(int(time.time() * 1000))
        return [f"KALSHI-ACCESS-KEY: {KEY_ID}",
                f"KALSHI-ACCESS-SIGNATURE: {_sign(self.k, ts, 'GET', WS_PATH)}",
                f"KALSHI-ACCESS-TIMESTAMP: {ts}"]

    def start(self): threading.Thread(target=self._run, daemon=True).start()

    def stop(self):
        self.stop_flag.set()
        try:
            if self.ws: self.ws.close()
        except Exception: pass

    def _run(self):
        backoff = 1
        while not self.stop_flag.is_set():
            try:
                self.ready.clear()
                ws = websocket.create_connection(WS_URL, header=self._headers(), timeout=15)
                with self.lock:
                    self.ws, self.sids = ws, {}
                    self.subscribed, self.sub_pending = set(), False
                    self.prices.clear()      # fresh snapshot comes with the new subscription
                self.connect_count += 1
                backoff = 1
                self._sync()
                print(f"[ws] connected (#{self.connect_count})")
                self.ready.set()
                while not self.stop_flag.is_set():
                    raw = ws.recv()
                    if raw: self._on_message(raw)
            except Exception as e:
                if self.stop_flag.is_set(): break
                self.ready.clear()
                try:
                    if self.ws: self.ws.close()
                except Exception: pass
                with self.lock:
                    self.ws, self.sids = None, {}
                    self.subscribed, self.sub_pending = set(), False
                print(f"[ws] disconnected ({e}) — reconnecting in {backoff:.0f}s")
                time.sleep(backoff)
                backoff = min(backoff * 2, 30)

    def _send(self, obj):
        with self.send_lock:
            if self.ws: self.ws.send(json.dumps(obj))

    def _subscribe(self, tickers):
        self.subscribed, self.sub_pending = set(tickers), True
        for ch in self.CHANNELS:
            p = {"channels": [ch], "market_tickers": sorted(tickers)}
            if ch == "ticker": p["send_initial_snapshot"] = True
            self._send({"id": self._cmd_id(), "cmd": "subscribe", "params": p})

    def _on_message(self, raw):
        try: msg = json.loads(raw)
        except (json.JSONDecodeError, TypeError): return
        mtype = msg.get("type")
        m = msg.get("msg") or {}
        if mtype == "subscribed":
            if m.get("channel") in self.CHANNELS: self.sids[m["channel"]] = m.get("sid")
            if len(self.sids) == len(self.CHANNELS) and self.sub_pending:
                self.sub_pending = False
                self._sync()      # catch up on markets added while waiting
        elif mtype == "ticker":
            t = m.get("market_ticker")
            yb, ya = px_c(m, "yes_bid"), px_c(m, "yes_ask")
            if not t or yb is None or ya is None: return
            with self.lock:
                self.prices[t] = (yb, ya, time.time())
            if self.on_tick:
                try: self.on_tick(t, yb, ya)
                except Exception as e: print(f"[tick-error] {e}")
        elif mtype == "trade":
            t = m.get("market_ticker")
            yp = px_c(m, "yes_price")
            n = m.get("count", m.get("count_fp"))
            try: n = int(round(float(n)))
            except (TypeError, ValueError): n = None
            side = m.get("taker_side")
            if not t or yp is None or not n or side not in ("yes", "no"): return
            if self.on_trade:
                try: self.on_trade(t, yp, n, side)
                except Exception as e: print(f"[trade-error] {e}")
        elif mtype == "error":
            print(f"[ws-error] {m}")

    def track(self, tickers):
        with self.lock:
            self.tracked = set(tickers)
        self._sync()

    def _sync(self):
        """Bring the live subscription in line with self.tracked."""
        with self.lock:
            if self.ws is None or self.sub_pending: return
            want = set(self.tracked)
            if len(self.sids) < len(self.CHANNELS):
                if want: self._subscribe(want)
                return
            add, remove = want - self.subscribed, self.subscribed - want
            self.subscribed = want
        for ch in self.CHANNELS:
            sid = self.sids.get(ch)
            if add:
                self._send({"id": self._cmd_id(), "cmd": "update_subscription",
                            "params": {"sid": sid, "market_tickers": sorted(add), "action": "add_markets"}})
            if remove:
                self._send({"id": self._cmd_id(), "cmd": "update_subscription",
                            "params": {"sid": sid, "market_tickers": sorted(remove), "action": "delete_markets"}})

    def get(self, ticker):
        """(yes_bid, yes_ask) while the feed is connected, else None. Kalshi only
        sends a ticker update when the price changes, so a quiet market's last
        price is still current as long as the connection is up."""
        if not self.ready.is_set(): return None
        with self.lock:
            d = self.prices.get(ticker)
        return (d[0], d[1]) if d else None

def make_feed(k):
    f = PriceFeed(k); f.start(); return f

# ---------------- the bot ----------------
class Position:
    def __init__(self, yes, px, s2c, n):
        self.yes, self.px, self.s2c, self.n = yes, px, s2c, n
        self.iso = now_iso()
        self.low = px          # lowest bid seen while held (logged for later analysis)

class MarketState:
    def __init__(self):
        self.pos = None        # the open position, if any
        self.trades = 0        # positions opened in this market

def main():
    print(f"{BOT_NAME} — PAPER ONLY (no orders are sent)")
    print(f"markets: {', '.join(asset(s) for s in SERIES)}")
    print(f"rules: buy window = minute {(900 - BUY_FROM_SEC) // 60:.0f} to {(900 - BUY_UNTIL_SEC) // 60:.0f} of each market · "
          f"BUY {CONTRACTS} of a side whose ask is {BUY_MIN_C}-{BUY_MAX_C}c (at the ask) · "
          f"SELL when its bid is {SELL_AT_C}c+ (at the bid) · else held to settlement · "
          f"repeat while the window is open · no stops, no reversals · no buy if spread > {MAX_SPREAD_C}c · "
          f"Kalshi taker fees")
    k = load_key()
    sess = requests.Session()
    _ad = requests.adapters.HTTPAdapter(pool_maxsize=len(SERIES) + 10)
    sess.mount("https://", _ad)
    lock = threading.RLock()
    markets, rest_px, state = {}, {}, {}
    settle_due, settled = {}, set()
    run = {"net": 0, "closed": 0, "sold": 0}

    def open_csv(path, cols):
        new = not os.path.exists(path)
        fh = open(path, "a", newline="")
        w = csv.writer(fh)
        if new: w.writerow(cols); fh.flush()
        return fh, w
    tfh, tw = open_csv(TRADES_CSV, TRADE_COLS)
    kfh, kw = open_csv(TICK_CSV, TICK_COLS)
    pfh, pw = open_csv(PRINTS_CSV, PRINT_COLS)
    sfh, sw = open_csv(SETTLE_CSV, SETTLE_COLS)
    io_lock = threading.Lock()

    def s2c_of(t):
        m = markets.get(t)
        return (m["close"] - now()).total_seconds() if m else None

    def say(msg): print(f"[{now():%H:%M:%S}] {msg}")

    def prices(t):
        return feed.get(t) or rest_px.get(t)

    def record(t, st, p, sell_px, sell_s2c, result, reason):
        fees = taker_fee_c(p.n, p.px) + (taker_fee_c(p.n, sell_px) if reason == "sold" else 0)
        net = (sell_px - p.px) * p.n - fees
        run["net"] += net; run["closed"] += 1
        if reason == "sold": run["sold"] += 1
        with io_lock:
            tw.writerow([t, asset(t), "PAPER", st.trades, "YES" if p.yes else "NO", p.iso,
                         round(p.s2c, 1), p.px, now_iso() if reason == "sold" else "",
                         round(sell_s2c, 1) if sell_s2c is not None else "", sell_px, result,
                         reason, p.low, fees, net, run["net"]])
            tfh.flush()
        return net

    def evaluate(t):
        px = prices(t)
        s2c = s2c_of(t)
        if px is None or s2c is None or s2c <= DEADBAND_SEC: return
        yb, ya = px
        if not (0 <= yb < ya <= 100): return
        with lock:
            st = state.setdefault(t, MarketState())
            p = st.pos
            if p:                                           # holding: sell at 45c+
                hb = yb if p.yes else 100 - ya
                p.low = min(p.low, hb)
                if hb >= SELL_AT_C:
                    st.pos = None
                    net = record(t, st, p, hb, s2c, "", "sold")
                    say(f"SELL {p.n}x {'YES' if p.yes else 'NO'} @ {hb}c (bought {p.px}c)  net {net:+.0f}c  "
                        f"{asset(t)} {t[-7:]}  {s2c:.0f}s left  run {run['net']:+.0f}c")
                return
            if not (BUY_UNTIL_SEC <= s2c <= BUY_FROM_SEC): return
            if ya - yb > MAX_SPREAD_C: return
            cands = [(ask, yes) for yes, ask in ((True, ya), (False, 100 - yb))
                     if BUY_MIN_C <= ask <= BUY_MAX_C]
            if not cands: return
            if len(cands) == 2 and cands[0][0] == cands[1][0]: return   # both sides same price: wait
            ask, yes = min(cands)                                       # the cheaper side
            st.pos = Position(yes, ask, s2c, CONTRACTS)
            st.trades += 1
            say(f"BUY {CONTRACTS}x {'YES' if yes else 'NO'} @ {ask}c  {asset(t)} {t[-7:]}  "
                f"{s2c:.0f}s left  [trade #{st.trades} in this market]")

    def on_tick(t, yb, ya):
        s2c = s2c_of(t)
        with io_lock:
            kw.writerow([now_iso(), t, asset(t), yb, ya, round(s2c, 1) if s2c is not None else ""])
        evaluate(t)

    def on_trade(t, yes_price, count, taker_side):
        s2c = s2c_of(t)
        with io_lock:
            pw.writerow([now_iso(), t, asset(t), yes_price, count, taker_side,
                         round(s2c, 1) if s2c is not None else ""])

    def finish(t, result):
        with lock:
            st = state.get(t)
            if not st or not st.pos: return
            p, st.pos = st.pos, None
            won = (result == "yes") == p.yes
            net = record(t, st, p, 100 if won else 0, None, result, "settled")
        say(f"SETTLED {'WIN ' if won else 'LOSS'} {p.n}x {'YES' if p.yes else 'NO'} bought {p.px}c  "
            f"net {net:+.0f}c  {asset(t)} {t[-7:]}  run {run['net']:+.0f}c "
            f"({run['closed']} positions, {run['sold']} sold at {SELL_AT_C}c+)")

    global FEED
    feed = FEED = make_feed(k)
    feed.on_tick = on_tick
    feed.on_trade = on_trade
    feed.ready.wait(timeout=10)

    last_refresh, last_flush = 0, time.time()
    try:
        while True:
            if time.time() - last_refresh >= MARKETS_REFRESH_SEC:
                last_refresh = time.time()
                try:
                    ms = fetch_open(sess, k)
                except Exception as e:
                    print(f"[rest] {e}"); ms = None
                if ms is not None:
                    for m in ms:
                        t, ct = m.get("ticker"), parse_iso(m.get("close_time"))
                        if not t or not ct or t in settled: continue
                        markets[t] = {"close": ct, "strike": m.get("floor_strike")}
                        yb, ya = px_c(m, "yes_bid"), px_c(m, "yes_ask")
                        if yb is not None and ya is not None: rest_px[t] = (yb, ya)
                    feed.track({t for t in markets if 0 < (s2c_of(t) or 0) <= 15 * 60 + 60})
            for t in list(markets):
                s2c = s2c_of(t)
                if s2c is None: continue
                if s2c > -5:
                    evaluate(t)
                elif t not in settle_due:
                    settle_due[t] = [time.time(), time.time() + SETTLE_GIVEUP_SEC]
            checked = 0
            for t, due in list(settle_due.items()):
                if checked >= 4: break
                if time.time() < due[0]: continue
                checked += 1
                m = fetch_market(sess, k, t)
                res = m.get("result")
                if res in ("yes", "no"):
                    with io_lock:
                        sw.writerow([now_iso(), t, asset(t), m.get("close_time"), res,
                                     m.get("floor_strike"), m.get("expiration_value"), m.get("status")])
                        sfh.flush()
                    finish(t, res)
                    settled.add(t)
                    del settle_due[t]; markets.pop(t, None); rest_px.pop(t, None)
                    with lock: state.pop(t, None)
                elif time.time() > due[1]:
                    settled.add(t)
                    del settle_due[t]; markets.pop(t, None)
                else:
                    due[0] = time.time() + SETTLE_RETRY_SEC
            if time.time() - last_flush > 2:
                last_flush = time.time()
                with io_lock:
                    kfh.flush(); pfh.flush()
            time.sleep(LOOP_SEC)
    except KeyboardInterrupt:
        print(f"[stopped] net {run['net']:+.0f}c over {run['closed']} closed positions "
              f"({run['sold']} sold at {SELL_AT_C}c+)")
    finally:
        feed.stop()
        with io_lock:
            for fh in (tfh, kfh, pfh, sfh): fh.close()

FEED = None


In [ ]:
# RUN BOT (PAPER ONLY). Stop with the stop button; totals print on stop.
main()
